# 01 — Data Processing

Loads `data/ITEmp.xlsx`, audits the raw fields against the data dictionary, cleans them and writes the analysis file.

**What this notebook does**

1. Defines the variable taxonomy and the plausible range of every field.
2. Coerces every analytic column to numeric, so that text entered by mistake becomes a missing value rather than a silent string.
3. Flags values that contradict the data dictionary (dummy variables outside {0,1}, salaries of 1,111 USD/hour, 1,001 years of education).
4. Imputes the remaining gaps — median for continuous variables, mode for dummies.
5. Adds the derived variables the models need and writes `ITEmp_Cleaned.csv`, `ITEmp_CompleteCase.csv` and a cleaning audit.

No value is ever *guessed*: anything irreconcilable with the data dictionary is treated as missing and then imputed by a documented rule.

> *One of three notebooks: `01_data_processing`, `02_eda_visualisations` and `03_hypothesis_regression`. Run them in order from the `notebooks/` directory.*

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_PATH = ROOT / "data" / "ITEmp.xlsx"
SHEET = "Salary"
OUT_CLEAN = ROOT / "ITEmp_Cleaned.csv"
OUT_COMPLETE = ROOT / "ITEmp_CompleteCase.csv"
OUT_LOG = ROOT / "output_tables" / "cleaning_log.csv"
OUT_SUMMARY = ROOT / "output_tables" / "cleaning_summary.txt"

## 1. Variable taxonomy (from the "Data Description" sheet of ITEmp.xlsx)

First we record what each column means and what values it can legitimately take. The dependent variable is `sal` (average hourly salary). The remaining variables fall into three groups: continuous/count measures, binary dummies, and the derived terms built later.

The `VALID_RANGE` dictionary encodes the data dictionary: a binary indicator can only be 0 or 1, education cannot exceed 25 years, and so on. Anything outside these bounds is impossible for an IT professional and is treated as a data-entry error.

In [2]:
ID_COL = "id"
DV = "sal"

BINARY_VARS = [
    "nonprev", "female", "married", "metro",
    "norcen", "south", "west",
    "itreq", "itprog", "itqual", "itarch",
    "itai", "itclient", "itrd", "ittech",
    "wrkhm",
]

CONTINUOUS_VARS = ["sal", "edu", "exper", "ten", "dep", "momed", "daded", "sib"]

# Plausible ranges implied by the data dictionary.  Values outside these
# are impossible for an IT professional and are treated as missing.
VALID_RANGE = {
    "sal":   (0.0, 100.0),   # average hourly salary, strictly positive
    "edu":   (1.0, 25.0),    # years of formal education
    "exper": (0.0, 60.0),    # years of professional experience
    "ten":   (0.0, 60.0),    # years in the current company
    "dep":   (0.0, 10.0),    # number of dependents
    "momed": (0.0, 25.0),    # mother's years of education
    "daded": (0.0, 25.0),    # father's years of education
    "sib":   (0.0, 10.0),    # number of siblings
}

# Region reference category is the eastern province (no dummy present).
REGION_DUMMIES = ["norcen", "south", "west"]
IT_FUNCTION_DUMMIES = ["itreq", "itprog", "itqual", "itarch"]
IT_DIVISION_DUMMIES = ["itrd", "ittech"]

ALL_VARS = [DV] + CONTINUOUS_VARS[1:] + BINARY_VARS

## 2. Loading and coercion

`pd.to_numeric(errors="coerce")` is the workhorse here: any non-numeric text (`SS0`, `aaa`, `acb`, `ab`, `b`, `s`) becomes `NaN` instead of silently poisoning the arithmetic. We then apply the range rules and set every violation to `NaN` too.

In [3]:
def load_raw(path: Path = RAW_PATH) -> pd.DataFrame:
    """Load the raw Salary sheet exactly as supplied."""
    return pd.read_excel(path, sheet_name=SHEET)


def coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Force every analytic column to numeric; non-numeric text -> NA.

    ``pd.to_numeric(errors="coerce")`` turns entries such as 'SS0', 'aaa',
    'acb', 'ab', 'b' and 's' into NaN so they can be audited like any other
    missing cell.
    """
    out = df.copy()
    for col in ALL_VARS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def apply_valid_ranges(df: pd.DataFrame) -> pd.DataFrame:
    """Set out-of-range continuous values and non-{0,1} dummies to NA."""
    out = df.copy()
    for col, (lo, hi) in VALID_RANGE.items():
        out.loc[(out[col] < lo) | (out[col] > hi), col] = np.nan
    for col in BINARY_VARS:
        out.loc[~out[col].isin([0, 1]), col] = np.nan
    return out

## 3. Auditing

The audit distinguishes three sources of missingness — values missing in the raw file, non-numeric text, and out-of-range numbers — so the report can state precisely how much of the file was unusable and why. This table is what feeds Table 1 of the written report.

In [4]:
def build_cleaning_log(raw: pd.DataFrame, coerced: pd.DataFrame,
                       ranged: pd.DataFrame) -> pd.DataFrame:
    """One row per column: text errors, range errors, original missingness."""
    rows = []
    for col in ALL_VARS:
        raw_series = raw[col]
        text_errors = int(coerced[col].isna().sum() - raw_series.isna().sum())
        range_errors = int(ranged[col].isna().sum() - coerced[col].isna().sum())
        rows.append({
            "variable": col,
            "role": ("dependent" if col == DV else
                     "binary" if col in BINARY_VARS else "continuous"),
            "raw_missing": int(raw_series.isna().sum()),
            "non_numeric_errors": max(text_errors, 0),
            "out_of_range_errors": max(range_errors, 0),
            "total_set_to_missing": int(ranged[col].isna().sum()),
        })
    return pd.DataFrame(rows)


def impute(df: pd.DataFrame) -> pd.DataFrame:
    """Median-impute continuous columns and mode-impute binary columns."""
    out = df.copy()
    for col in CONTINUOUS_VARS:
        out[col] = out[col].fillna(out[col].median())
    for col in BINARY_VARS:
        out[col] = out[col].fillna(out[col].mode().iloc[0]).astype(int)
    return out


def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Add transformations used by the modelling stage."""
    out = df.copy()
    out["log_sal"] = np.log(out["sal"])
    out["exper_sq"] = out["exper"] ** 2
    out["ten_sq"] = out["ten"] ** 2
    out["edu_sq"] = out["edu"] ** 2
    return out

## 4. Pipeline

Finally we impute and write the outputs. Two datasets are produced: the imputed analysis file (526 rows, used for the main models) and a complete-case file (482 rows, used as a robustness check so that no conclusion rests on the imputation).

In [5]:
def run(verbose: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = load_raw()
    coerced = coerce_numeric(raw)
    ranged = apply_valid_ranges(coerced)
    log = build_cleaning_log(raw, coerced, ranged)

    clean = add_derived(impute(ranged))
    clean = clean.sort_values(ID_COL).reset_index(drop=True)

    # complete-case (listwise) version: rows with no missing *analytic* value
    complete = add_derived(ranged.dropna(subset=ALL_VARS))
    complete = complete.sort_values(ID_COL).reset_index(drop=True)

    OUT_CLEAN.parent.mkdir(parents=True, exist_ok=True)
    clean.to_csv(OUT_CLEAN, index=False)
    complete.to_csv(OUT_COMPLETE, index=False)
    log.to_csv(OUT_LOG, index=False)

    n_cells_bad = int(log["non_numeric_errors"].sum() + log["out_of_range_errors"].sum())
    n_cells_imputed = int(ranged[ALL_VARS].isna().sum().sum())
    summary = "\n".join([
        "DOM207 MP2 - cleaning summary",
        "=" * 45,
        f"Raw rows                       : {len(raw)}",
        f"Raw columns (incl. id)         : {raw.shape[1]}",
        f"Non-numeric data-entry errors  : {int(log['non_numeric_errors'].sum())} cells",
        f"Out-of-range data-entry errors : {int(log['out_of_range_errors'].sum())} cells",
        f"Total invalid cells set to NA  : {n_cells_bad}",
        f"Pre-existing missing cells     : {int(log['raw_missing'].sum())} cells",
        f"Cells imputed (median/mode)    : {n_cells_imputed} "
        f"({n_cells_imputed / (len(raw) * len(ALL_VARS)) * 100:.2f}% of analytic cells)",
        f"Rows in imputed analysis file  : {len(clean)}",
        f"Rows in complete-case file     : {len(complete)}",
        "",
    ])
    OUT_SUMMARY.write_text(summary)

    if verbose:
        print(summary)
        print(log.to_string(index=False))
    return clean, log


if __name__ == "__main__":
    run()

DOM207 MP2 - cleaning summary
Raw rows                       : 526
Raw columns (incl. id)         : 25
Non-numeric data-entry errors  : 8 cells
Out-of-range data-entry errors : 21 cells
Total invalid cells set to NA  : 29
Pre-existing missing cells     : 22 cells
Cells imputed (median/mode)    : 51 (0.40% of analytic cells)
Rows in imputed analysis file  : 526
Rows in complete-case file     : 482

variable       role  raw_missing  non_numeric_errors  out_of_range_errors  total_set_to_missing
     sal  dependent            2                   1                    4                     7
     edu continuous            1                   1                    5                     7
   exper continuous            2                   0                    0                     2
     ten continuous            0                   0                    0                     0
     dep continuous            0                   0                    0                     0
   momed continuous    

## Outputs

The audit trail. `cleaning_log.csv` shows, per variable, how many cells were missing in the raw file, how many were non-numeric, how many were out of range, and how many were therefore imputed.

In [6]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(ROOT / 'output_tables' / 'cleaning_log.csv'))
print((ROOT / 'output_tables' / 'cleaning_summary.txt').read_text())

,variable,role,raw_missing,non_numeric_errors,out_of_range_errors,total_set_to_missing
0,sal,dependent,2,1,4,7
1,edu,continuous,1,1,5,7
2,exper,continuous,2,0,0,2
3,ten,continuous,0,0,0,0
4,dep,continuous,0,0,0,0
5,momed,continuous,3,1,1,5
6,daded,continuous,2,0,1,3
7,sib,continuous,1,0,4,5
8,nonprev,binary,0,0,0,0
9,female,binary,0,0,0,0


DOM207 MP2 - cleaning summary
Raw rows                       : 526
Raw columns (incl. id)         : 25
Non-numeric data-entry errors  : 8 cells
Out-of-range data-entry errors : 21 cells
Total invalid cells set to NA  : 29
Pre-existing missing cells     : 22 cells
Cells imputed (median/mode)    : 51 (0.40% of analytic cells)
Rows in imputed analysis file  : 526
Rows in complete-case file     : 482

